In [5]:
from pathlib import Path
import polars as pl
import plotly.express as px

# Resolver la raíz del proyecto estando dentro de /notebooks/
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_CSV_PATH = BASE_DIR / "data" / "raw" / "Lluvia_2026_v2.csv"

print(f"Ruta apuntada: {RAW_CSV_PATH}")
print(f"¿Existe el archivo?: {RAW_CSV_PATH.exists()}")

Ruta apuntada: /Users/pablo/Documents/lluvias-chile-2026/data/raw/Lluvia_2026_v2.csv
¿Existe el archivo?: True


In [6]:
df_raw = pl.read_csv(
    RAW_CSV_PATH,
    separator=";",
    schema_overrides={
        "hora": pl.Int32,
        "fecha": pl.Float64,
        "lluvia_mm": pl.Float64,
    },
)

# Salida ad hoc: inspección de estructura inicial
df_raw.glimpse()

Rows: 56
Columns: 3
$ hora      <i32> 2, 5, 8, 11, 14, 17, 20, 23, 2, 5
$ fecha     <f64> 46219.0, 46219.0, 46219.0, 46219.0, 46219.0, 46219.0, 46219.0, 46219.0, 46220.0, 46220.0
$ lluvia_mm <f64> 0.2, 0.0, 0.8, 3.1, 6.0, 10.0, 17.0, 18.0, 12.0, 10.0



In [7]:
df_base = (
    df_raw.with_columns(
        fecha_date=pl.date(1899, 12, 30) + pl.duration(days=pl.col("fecha"))
    )
    .with_columns(
        datetime=pl.col("fecha_date").dt.combine(
            pl.time(hour=pl.col("hora"),
                    minute=0, second=0)
        )
    )
    .sort("datetime")
)

# Salida ad hoc: verificar la conversión temporal
df_base.select(["fecha", "fecha_date", "hora", "datetime", "lluvia_mm"]).head(5)

fecha,fecha_date,hora,datetime,lluvia_mm
f64,date,i32,datetime[μs],f64
46219.0,2026-07-16,2,2026-07-16 02:00:00,0.2
46219.0,2026-07-16,5,2026-07-16 05:00:00,0.0
46219.0,2026-07-16,8,2026-07-16 08:00:00,0.8
46219.0,2026-07-16,11,2026-07-16 11:00:00,3.1
46219.0,2026-07-16,14,2026-07-16 14:00:00,6.0


In [8]:
target_hours = [6] + list(range(12, 97, 12))

rolling_exprs = [
    pl.col("lluvia_mm")
    .rolling_mean(window_size=(h // 3),
                  min_samples=(h // 3))
    .alias(f"ma_{h}h")
    for h in target_hours
]

df_matrix = df_base.with_columns(
    [pl.col("lluvia_mm").cum_sum().alias("sum_acum")] + rolling_exprs
)

# Salida ad hoc: revisar la matriz resultante
df_matrix.head(10)

hora,fecha,lluvia_mm,fecha_date,datetime,sum_acum,ma_6h,ma_12h,ma_24h,ma_36h,ma_48h,ma_60h,ma_72h,ma_84h,ma_96h
i32,f64,f64,date,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2,46219.0,0.2,2026-07-16,2026-07-16 02:00:00,0.2,null,null,null,null,null,null,null,null,null
5,46219.0,0.0,2026-07-16,2026-07-16 05:00:00,0.2,0.1,null,null,null,null,null,null,null,null
8,46219.0,0.8,2026-07-16,2026-07-16 08:00:00,1.0,0.4,null,null,null,null,null,null,null,null
11,46219.0,3.1,2026-07-16,2026-07-16 11:00:00,4.1,1.95,1.025,null,null,null,null,null,null,null
14,46219.0,6.0,2026-07-16,2026-07-16 14:00:00,10.1,4.55,2.475,null,null,null,null,null,null,null
17,46219.0,10.0,2026-07-16,2026-07-16 17:00:00,20.1,8.0,4.975,null,null,null,null,null,null,null
20,46219.0,17.0,2026-07-16,2026-07-16 20:00:00,37.1,13.5,9.025,null,null,null,null,null,null,null
23,46219.0,18.0,2026-07-16,2026-07-16 23:00:00,55.1,17.5,12.75,6.8875,null,null,null,null,null,null
2,46220.0,12.0,2026-07-17,2026-07-17 02:00:00,67.1,15.0,14.25,8.3625,null,null,null,null,null,null


In [9]:
ma_cols = [f"ma_{h}h" for h in target_hours]

df_plot = df_matrix.with_columns(
    pl.col("datetime").dt.strftime("%Y-%m-%d %H:00").alias("datetime_str")
)

timestamps = df_plot["datetime_str"].to_list()
matrix_values = df_plot.select(ma_cols).to_numpy().T

fig = px.imshow(
    matrix_values,
    labels=dict(x="Tiempo",
                y="Ventana Temporal",
                color="Precipitación Prom. (mm)"),
    x=timestamps,
    y=[f"{h}h" for h in target_hours],
    color_continuous_scale="Reds",
    title="Mapa de Calor: Acumulación Temporal de Lluvias (mm) por Ventana Móvil y Variable",
    aspect="auto",
)

fig.update_xaxes(side="bottom",
                 tickangle=-45)
fig.show()

In [10]:
# Celda 6: Exportación de matriz completa a Parquet
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True,
                    exist_ok=True)

OUTPUT_PARQUET_PATH = PROCESSED_DIR / "lluvia_2026_matrix.parquet"

# Exportar preservando nulos (ventanas de inicialización) y tipos de datos
df_matrix.write_parquet(OUTPUT_PARQUET_PATH)